[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_69_OSS_Growth_Round_Two.ipynb)

# Lesson 69 — agent-bench: OSS Growth, Round Two (README, Badges, Contributor Funnel & Cross-Promotion)

**Phase 7 · Lesson 5 of ~6 (tentative)** — *A Second Flagship OSS Tool*

`agent-bench` now has a working core (L65), a real plugin system (L66), async parallel execution (L67), and a packaged, publishable CLI (L68). All of that is still **invisible** — exactly the problem L60 solved for `paper-distiller` back in Phase 6. Today we do the same job again, for a second package.

The lazy move would be to copy L60's notebook, find-and-replace `paper-distiller` with `agent-bench`, and call it done. We won't do that, for one specific reason: **this isn't your first repo anymore.** You now run a small *portfolio* of two OSS packages under one GitHub identity, and `agent-bench` has a structural feature paper-distiller never had — a real third-party plugin system (L66's `entry_points()` discovery, proven live with `agent-bench-shell-plugin`). Both of those facts change the shape of the growth funnel in ways that are genuinely new material, not a repeat of L60.

**What's actually new this lesson (not in L60):**
1. **Portfolio-level concerns** — org-wide community health files (GitHub's special `.github` repo), cross-promotion between two repos, and the "don't launch both in the same week" trap.
2. **Plugin-shaped contribution funnel** — for a plugin architecture, the lowest-friction contribution isn't a PR to core, it's *publishing your own plugin package*. CONTRIBUTING.md and issue templates need a dedicated path for that.
3. **A new metric** — external contribution rate (L60) still matters, but a plugin ecosystem adds a second number worth tracking: *plugins published by people who aren't you.*

### Phase 7 roadmap (tentative, adaptive)

| Lesson | Topic | Status |
|---|---|---|
| L65 | Phase 7 Kickoff — plugin registry pattern | Done |
| L66 | Real Plugins — `ShellEnv` + `entry_points()` | Done |
| L67 | Async Parallel Execution — `asyncio.gather` + `Semaphore` | Done |
| L68 | CLI Polish + PyPI Packaging | Done |
| **L69** | **OSS Growth, Round Two — README, badges, contributor funnel, cross-promotion** | **Today** |
| L70 | Launch Day — Phase 7 capstone, actually publish `agent-bench` | Next |


## The Concept: Growing a Portfolio, Not Just a Repo

L60 covered the acquisition funnel (Discovery → First Impression → Activation → Contribution → Retention) in depth — that still applies unchanged and we won't re-derive it. What's different today is the *context* the funnel operates in.

| | L60 (`paper-distiller`, first repo) | L69 (`agent-bench`, second repo) |
|---|---|---|
| Community health files | Own copy of CODE_OF_CONDUCT.md, FUNDING.yml per repo | Candidate for **org-level defaults** via a special `.github` repository — GitHub falls back to it for any repo that doesn't define its own copy |
| README positioning | Only project — no competition for attention | Must state its relationship to `paper-distiller` (same author, different problem) or visitors wonder if they're duplicates |
| Contribution funnel | One path: PR to core | **Two paths**: PR to core, or publish an independent plugin package (L66) — the second is lower-friction and needs its own doc section and issue template |
| Launch sequencing | No prior launch to conflict with | Must NOT launch the same week as a `paper-distiller` push — audience attention is finite and shared |
| Badge staleness risk | One CI badge to keep green | Two independent CI badges across two repos, silently drifting apart if only one gets attention |

The one genuinely new mechanic: **GitHub's org-level `.github` repository.** If you create a repository literally named `.github` under your account/org, files like `CODE_OF_CONDUCT.md`, `SUPPORT.md`, and `FUNDING.yml` placed in its root (or `.github/` subdir) become the **default** for every repo in the org that doesn't define its own copy. It's the same mechanism as a `.gitignore` fallback — repo-local always wins, org-level is what shows up when a repo is silent. Worth doing once you have 2+ repos; not worth it for a single repo, which is why L60 didn't cover it.


In [ ]:
# Setup — no LLM calls needed today; this is pure engineering artifacts, same as L60.
!pip install pyyaml -q

import os
from pathlib import Path

AGENT_BENCH_ROOT = Path("/content/agent_bench_repo")
AB_GITHUB_DIR = AGENT_BENCH_ROOT / ".github"
AB_ISSUE_TEMPLATE_DIR = AB_GITHUB_DIR / "ISSUE_TEMPLATE"

ORG_GITHUB_ROOT = Path("/content/dot_github_repo")  # the org-level ".github" repo scaffold

for d in [AGENT_BENCH_ROOT, AB_GITHUB_DIR, AB_ISSUE_TEMPLATE_DIR, ORG_GITHUB_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print("Scaffold created:")
print(f"  {AGENT_BENCH_ROOT}  (per-repo files)")
print(f"  {ORG_GITHUB_ROOT}  (org-wide default files)")


## The README: Same Anatomy, One New Section

The strict ordering from L60 is unchanged — hook → badges → quickstart → comparison table → architecture → usage → contributing → license — and still the right call; skipping straight to a rewrite of that ordering here would be regressive, not incremental.

What's new: a **Plugin Gallery** section. `agent-bench` isn't just a benchmark harness, it's a benchmark harness with a real `entry_points()`-based plugin mechanism (L66). A visitor deciding whether to trust the project needs to see, at a glance, what environments already exist — built-in and third-party — before they commit to writing task suites against it. This is the README section paper-distiller structurally cannot have, because it has no plugin system.


In [ ]:
README_MD = r'''# agent-bench

**A pluggable benchmark harness for LLM agents — Task -> Environment -> Agent -> Trajectory -> Scorer, with pass@k, async parallel execution, and a real third-party plugin system.**

[![CI](https://github.com/gouravkhanijoe/agent-bench/actions/workflows/ci.yml/badge.svg)](https://github.com/gouravkhanijoe/agent-bench/actions/workflows/ci.yml)
[![PyPI version](https://img.shields.io/pypi/v/agent-bench.svg)](https://pypi.org/project/agent-bench/)
[![Python versions](https://img.shields.io/pypi/pyversions/agent-bench.svg)](https://pypi.org/project/agent-bench/)
[![License](https://img.shields.io/github/license/gouravkhanijoe/agent-bench.svg)](https://github.com/gouravkhanijoe/agent-bench/blob/main/LICENSE)
[![Downloads](https://img.shields.io/pypi/dm/agent-bench.svg)](https://pypi.org/project/agent-bench/)

> Also building **[paper-distiller](https://github.com/gouravkhanijoe/paper-distiller)** — turns arXiv papers into practitioner digests. Different problem, same author, same quality bar. `agent-bench` can (and eventually will, see Issues) benchmark paper-distiller's own extraction agent.

## Why agent-bench

Ad-hoc "eyeball the transcript" evals and one-off pytest scripts don't scale past a handful of tasks, don't measure pass@k, and don't compose with anyone else's environments. `agent-bench` gives you a small stable core (Task/Environment/Agent/Trajectory/Scorer) plus a plugin registry, so adding a new benchmark environment never touches core code.

## 30-second quickstart

```bash
pip install agent-bench
agent-bench list-environments
agent-bench run --k 3 --concurrency 4
```

## agent-bench vs. the alternatives

| | Manual pytest script | Single-model benchmark script | agent-bench |
|---|---|---|---|
| pass@k support | No | Rarely | Built in (Chen et al. 2021 estimator) |
| Parallel execution | Manual | Manual | `--concurrency`, semaphore-bounded |
| New environment = | Edit the script | Edit the script | Install a plugin package, zero core edits |
| Third-party environments | No | No | Yes — `entry_points()` discovery |
| CLI + PyPI package | No | Sometimes | Yes |

## Architecture

```
Task -> Environment (get_environment(), fresh instance per call)
      -> Agent (MockAgent / ClaudeToolAgent / your own)
      -> Trajectory (steps recorded)
      -> Scorer (exact_match / unit_test / llm_judge / your own)
      -> TaskResult -> pass_at_k()
```

Registries (`ENVIRONMENT_REGISTRY`, `AGENT_REGISTRY`, `SCORER_REGISTRY`) are populated two ways: built-ins registered on import, third-party plugins registered via `entry_points()` (see Plugin Gallery below).

## Plugin Gallery

| Plugin | Kind | Source | Notes |
|---|---|---|---|
| `CalcEnv` | environment | built-in | Safe-eval calculator tool |
| `FileEnv` | environment | built-in | In-memory virtual filesystem, SWE-bench-shaped |
| `KnowledgeEnv` | environment | built-in | Fixed corpus + lookup tool |
| `agent-bench-shell-plugin` | environment (`ShellEnv`) | third-party (L66) | Sandboxed shell, default-deny command allowlist |
| *your plugin here* | — | — | See CONTRIBUTING.md -> "Writing a Plugin" |

## Usage

```bash
agent-bench run --k 3 --concurrency 4 --format json
agent-bench list-agents
agent-bench list-scorers
agent-bench list-plugins --kind environments
```

## Contributing

See [CONTRIBUTING.md](CONTRIBUTING.md) — includes a dedicated path for publishing your own environment/agent/scorer plugin without touching core.

## License

MIT
'''

readme_path = AGENT_BENCH_ROOT / "README.md"
readme_path.write_text(README_MD)
print(f"Wrote {readme_path} ({len(README_MD)} chars)")


## Badges: Still One Per Trust Question, Now Times Two Repos

The badge-to-question mapping from L60 is unchanged (CI/PyPI-version/Python-versions/License/Downloads). The new risk: two repos means **two independently-maintained badge sets**, and it's easy to keep one fresh while the other quietly goes stale (a broken CI badge nobody notices because attention is on the other repo). `badge_row()` from L60 is reused verbatim below — reuse, not rewrite, because the function itself didn't need to change, only where it's called from.

The genuinely new artifact: an **organization profile README**. Creating a repo named exactly `<username>/<username>` (e.g. `gouravkhanijoe/gouravkhanijoe`) makes its `README.md` render on the GitHub profile page — this is the natural place to cross-promote both packages instead of relying on each README's one-line mention of the other.


In [ ]:
def badge_row(pkg_name: str, gh_org: str, gh_repo: str) -> str:
    # Generate a verified badge row — call this whenever you rename the package or repo.
    # Reused unchanged from L60; the function didn't need to change, only its call site.
    base = f"https://github.com/{gh_org}/{gh_repo}"
    return "\n".join([
        f"[![CI]({base}/actions/workflows/ci.yml/badge.svg)]({base}/actions/workflows/ci.yml)",
        f"[![PyPI version](https://img.shields.io/pypi/v/{pkg_name}.svg)](https://pypi.org/project/{pkg_name}/)",
        f"[![Python versions](https://img.shields.io/pypi/pyversions/{pkg_name}.svg)](https://pypi.org/project/{pkg_name}/)",
        f"[![License](https://img.shields.io/github/license/{gh_org}/{gh_repo}.svg)]({base}/blob/main/LICENSE)",
        f"[![Downloads](https://img.shields.io/pypi/dm/{pkg_name}.svg)](https://pypi.org/project/{pkg_name}/)",
    ])

agent_bench_badges = badge_row("agent-bench", "gouravkhanijoe", "agent-bench")
paper_distiller_badges = badge_row("paper-distiller", "gouravkhanijoe", "paper-distiller")
print(agent_bench_badges)
assert agent_bench_badges.count("shields.io") + agent_bench_badges.count("actions/workflows") >= 4

ORG_PROFILE_README = f'''# Gourav Khanijoe

Building small, focused OSS tools for working with LLMs and agents.

### [paper-distiller](https://github.com/gouravkhanijoe/paper-distiller)
Turn any arXiv paper into a practitioner-friendly digest in under 30 seconds.

{paper_distiller_badges}

### [agent-bench](https://github.com/gouravkhanijoe/agent-bench)
A pluggable benchmark harness for LLM agents — pass@k, async execution, real plugin system.

{agent_bench_badges}
'''

org_profile_path = ORG_GITHUB_ROOT / "PROFILE_README.md"
org_profile_path.write_text(ORG_PROFILE_README)
print(f"\nWrote {org_profile_path} ({len(ORG_PROFILE_README)} chars)")


## The Contributor Funnel, Now With a Second On-Ramp

L60's funnel diagnosis (ambiguity at any step = silent drop-off, not a filed question) still holds. What changes for a plugin-shaped project: the **Contribution** stage forks into two paths, and the lower-friction one isn't the one most CONTRIBUTING.md files default to.

```
Discovery -> First Impression -> Activation (pip install, run it) -> Contribution -> Retention
                                                                        |
                                                          +-------------+-------------+
                                                          |                           |
                                                   PR to core repo          Publish your own
                                                   (needs review,           plugin package
                                                   maintainer bandwidth)    (independent PyPI
                                                                            release, entry_points
                                                                            table — no PR needed
                                                                            for the plugin to exist)
```

Most maintainers only document the left branch. The right branch is strictly lower-friction — L66 proved a third-party package can register into `ENVIRONMENT_REGISTRY` with zero code review from the core maintainer — and CONTRIBUTING.md should say so explicitly, with a copy-pasteable `pyproject.toml` snippet.


In [ ]:
CONTRIBUTING_MD = r'''# Contributing to agent-bench

Thanks for considering a contribution! There are two ways to contribute, and the second one is usually faster.

## Quick setup (for core contributions)

```bash
git clone https://github.com/gouravkhanijoe/agent-bench
cd agent-bench
pip install -e ".[dev]"
pytest                 # should be green before you change anything
```

## Path 1: PR to core

For bug fixes, new CLI flags, docs, or changes to `core.py`/`registry.py`/`runner_async.py`/`cli.py`. Needs maintainer review.

1. Check open issues labeled `good-first-issue` or `help-wanted`.
2. Open a PR referencing the issue.
3. Make sure `pytest` and `ruff` are clean, and CI passes.

## Path 2: Publish your own plugin (usually faster, no PR needed)

`agent-bench` discovers environments, agents, and scorers via Python entry points
(see L66) — you do **not** need a PR to core for your environment to show up in
`agent-bench list-plugins`. You need:

1. A separate installable package (own `pyproject.toml`).
2. A class implementing the `Environment` protocol (or `Agent` / `Scorer`).
3. An entry-point table pointing at it:

```toml
[project.entry-points."agent_bench.environments"]
my_env = "my_package.env:MyEnvironment"
```

4. `pip install -e .` locally to verify `agent-bench list-plugins` finds it, then publish
   to PyPI. That's the whole distribution story — see `agent-bench-shell-plugin` for a
   working reference implementation.
5. Optional but appreciated: open an issue using the **Plugin Submission** template so it
   gets listed in the README's Plugin Gallery.

## What a good PR looks like

- One logical change per PR
- Tests included for new behavior
- `ruff check .` and `mypy` clean
- Update CHANGELOG.md under `[Unreleased]`

## Code of Conduct

This project follows the org-wide [Code of Conduct](https://github.com/gouravkhanijoe/.github/blob/main/CODE_OF_CONDUCT.md).
'''

contributing_path = AGENT_BENCH_ROOT / "CONTRIBUTING.md"
contributing_path.write_text(CONTRIBUTING_MD)
print(f"Wrote {contributing_path} ({len(CONTRIBUTING_MD)} chars)")
assert "Path 2: Publish your own plugin" in CONTRIBUTING_MD


## Issue Templates: A New Template Type for Plugin Ecosystems

L60's `bug_report.yml` / `feature_request.yml` / `config.yml` pattern carries over unchanged. New this lesson: **`plugin_submission.yml`** — projects with a real plugin ecosystem (pytest, Sphinx, Terraform providers) commonly have a dedicated issue form maintainers use to catalog third-party plugins into the README's gallery, separate from bug reports or feature requests. Without it, plugin announcements get filed as generic "feature requests" and are easy to lose track of.


In [ ]:
BUG_REPORT_YML = r'''name: Bug Report
description: Something isn't working as expected
title: "[Bug]: "
labels: ["bug", "triage"]
body:
  - type: input
    id: version
    attributes:
      label: agent-bench version
      placeholder: "0.1.0"
    validations:
      required: true
  - type: textarea
    id: what-happened
    attributes:
      label: What happened?
      description: Include the exact `agent-bench` command you ran and the full traceback.
    validations:
      required: true
  - type: textarea
    id: expected
    attributes:
      label: What did you expect?
    validations:
      required: false
'''

FEATURE_REQUEST_YML = r'''name: Feature Request
description: Suggest an idea for agent-bench core
title: "[Feature]: "
labels: ["enhancement", "triage"]
body:
  - type: textarea
    id: problem
    attributes:
      label: What problem does this solve?
    validations:
      required: true
  - type: textarea
    id: proposal
    attributes:
      label: Proposed solution
    validations:
      required: false
'''

PLUGIN_SUBMISSION_YML = r'''name: Plugin Submission
description: Announce a third-party environment, agent, or scorer plugin for the README Plugin Gallery
title: "[Plugin]: "
labels: ["plugin", "triage"]
body:
  - type: input
    id: pkg-name
    attributes:
      label: PyPI package name
      placeholder: "agent-bench-my-plugin"
    validations:
      required: true
  - type: dropdown
    id: kind
    attributes:
      label: Plugin kind
      options:
        - environment
        - agent
        - scorer
    validations:
      required: true
  - type: input
    id: repo-url
    attributes:
      label: Source repository URL
    validations:
      required: true
  - type: textarea
    id: description
    attributes:
      label: One-line description for the gallery table
    validations:
      required: true
'''

CONFIG_YML = r'''blank_issues_enabled: false
contact_links:
  - name: Questions & Discussion
    url: https://github.com/gouravkhanijoe/agent-bench/discussions
    about: Ask questions or discuss ideas before filing an issue
'''

PR_TEMPLATE_MD = r'''## What does this PR do?


## Which path is this?
- [ ] Core contribution (Path 1)
- [ ] N/A — this is a plugin, announced via a Plugin Submission issue instead

## Checklist
- [ ] `pytest` passes locally
- [ ] `ruff check .` clean
- [ ] CHANGELOG.md updated under `[Unreleased]`
'''

templates = {
    "bug_report.yml": BUG_REPORT_YML,
    "feature_request.yml": FEATURE_REQUEST_YML,
    "plugin_submission.yml": PLUGIN_SUBMISSION_YML,
    "config.yml": CONFIG_YML,
}
for name, content in templates.items():
    p = AB_ISSUE_TEMPLATE_DIR / name
    p.write_text(content)
    print(f"Wrote {p}")

pr_path = AB_GITHUB_DIR / "PULL_REQUEST_TEMPLATE.md"
pr_path.write_text(PR_TEMPLATE_MD)
print(f"Wrote {pr_path}")


## Org-Level Community Health Files: the `.github` Repository

GitHub looks for `CODE_OF_CONDUCT.md`, `SUPPORT.md`, `SECURITY.md`, and `.github/FUNDING.yml` in two places, in this order: the repo itself, then — if absent — a special repository named exactly `.github` under the same user or org. That fallback repo is what makes a multi-repo portfolio maintainable: write the Code of Conduct once, and both `paper-distiller` and `agent-bench` inherit it unless they explicitly override it.

This is the one concrete payoff of having a second package: L60 had no reason to build this (one repo, nothing to share), and doing it a lesson early — before there were two real repos — would have been premature abstraction.


In [ ]:
CODE_OF_CONDUCT_MD = r'''# Contributor Covenant Code of Conduct

## Our Pledge

We as members, contributors, and leaders pledge to make participation in our
community a harassment-free experience for everyone, regardless of age, body
size, visible or invisible disability, ethnicity, sex characteristics, gender
identity and expression, level of experience, education, socio-economic status,
nationality, personal appearance, race, religion, or sexual identity and
orientation.

## Our Standards

Examples of behavior that contributes to a positive environment include
demonstrating empathy, being respectful of differing opinions, giving and
gracefully accepting constructive feedback, and focusing on what is best for
the community.

## Enforcement

Instances of abusive, harassing, or otherwise unacceptable behavior may be
reported to the project maintainer. All complaints will be reviewed and
investigated promptly and fairly.

This Code of Conduct is adapted from the [Contributor Covenant](https://www.contributor-covenant.org/), version 2.1.
'''

FUNDING_YML = r'''github: [gouravkhanijoe]
'''

SUPPORT_MD = r'''# Support

- **Bug or feature request?** Open an issue on the specific repo (`paper-distiller` or `agent-bench`).
- **General question?** Use that repo's Discussions tab.
- This Code of Conduct and Support policy apply org-wide (see the `.github` repo) unless a
  specific repo defines its own.
'''

# These live in the ORG_GITHUB_ROOT scaffold (the ".github" repo), not per-package repos.
org_files = {
    "CODE_OF_CONDUCT.md": CODE_OF_CONDUCT_MD,
    "SUPPORT.md": SUPPORT_MD,
}
(ORG_GITHUB_ROOT / ".github").mkdir(parents=True, exist_ok=True)
(ORG_GITHUB_ROOT / ".github" / "FUNDING.yml").write_text(FUNDING_YML)
for name, content in org_files.items():
    (ORG_GITHUB_ROOT / name).write_text(content)
    print(f"Wrote {ORG_GITHUB_ROOT / name}")
print(f"Wrote {ORG_GITHUB_ROOT / '.github' / 'FUNDING.yml'}")

# labels.yml stays per-repo (issue labels are repo-scoped in GitHub, no org-level fallback exists)
import yaml

LABELS = [
    {"name": "good-first-issue", "color": "7057ff", "description": "Small, scoped, no design decisions — start here"},
    {"name": "help-wanted", "color": "008672", "description": "Larger task, maintainer will support you"},
    {"name": "bug", "color": "d73a4a", "description": "Something isn't working"},
    {"name": "enhancement", "color": "a2eeef", "description": "New feature or request"},
    {"name": "documentation", "color": "0075ca", "description": "Improvements or additions to docs"},
    {"name": "plugin", "color": "fbca04", "description": "Third-party environment/agent/scorer plugin — announcement or discussion"},
    {"name": "triage", "color": "ededed", "description": "Needs maintainer review to categorize"},
]
labels_path = AGENT_BENCH_ROOT / "labels.yml"
labels_path.write_text(yaml.dump(LABELS, sort_keys=False))
print(f"Wrote {labels_path} ({len(LABELS)} labels, including 'plugin' — new vs L60's set)")


## Launch Strategy: Don't Cannibalize Your Own Attention

L60's soft-launch-then-hard-launch sequence (private feedback -> seed issues -> dogfood -> ONE hard-launch channel per week) still applies per repo. The new rule sits one level up: **never hard-launch `agent-bench` and `paper-distiller` in the same week.** A single person has one Hacker News post's worth of audience attention per week; splitting it between two announcements measurably weakens both, and readers who see two "look what I built" posts from the same account in quick succession discount both.

The other launch-sequencing detail specific to `agent-bench`: it target a different `awesome-*` list than paper-distiller did. `paper-distiller` fits `awesome-llm-apps`; `agent-bench` fits `awesome-llm-eval` / `awesome-agents` — different discovery channel, worth listing both packages separately rather than assuming one PR covers both.


In [ ]:
LAUNCH_CHECKLIST_MD = r'''# Launch Checklist — agent-bench v0.1.0

## Before any public post
- [ ] `pip install agent-bench` works on a clean machine/venv (verified for real, L68)
- [ ] README quickstart commands copy-pasted and verified end to end
- [ ] CI badge is green (not just present)
- [ ] 5+ issues labeled `good-first-issue`
- [ ] Plugin Gallery lists at least one real third-party plugin (`agent-bench-shell-plugin`, L66)
- [ ] CONTRIBUTING.md's "Path 2: Publish your own plugin" section is copy-paste-verified
- [ ] Cross-link added to paper-distiller's README (and vice versa)
- [ ] Confirmed: no paper-distiller hard-launch scheduled within +/- 1 week of this one

## Launch sequence
1. Soft launch privately (3-5 people)
2. Seed 5-10 good-first-issues from your own homework backlog (L65-L68)
3. Dogfood publicly for a few days (post to your own updates, not a big channel yet)
4. ONE hard-launch channel: Show HN **or** r/MachineLearning **or** Twitter/X — not stacked
5. Submit to `awesome-llm-eval` / `awesome-agents` (separate from paper-distiller's `awesome-llm-apps` PR)

## Metric to check one week later
- External contribution rate > 0 (see below)
- At least one plugin submission issue filed by someone who isn't you
'''

launch_path = AGENT_BENCH_ROOT / "LAUNCH_CHECKLIST.md"
launch_path.write_text(LAUNCH_CHECKLIST_MD)
print(f"Wrote {launch_path}")

# Cross-promotion snippet — the line that would be added to paper-distiller's own README.md
CROSS_PROMO_SNIPPET = (
    "> Also building **[agent-bench](https://github.com/gouravkhanijoe/agent-bench)** — "
    "a pluggable benchmark harness for LLM agents. Could eventually benchmark this project's "
    "own extraction agent (flagged as homework back in L64)."
)
print(f"\nSnippet to add to paper-distiller/README.md:\n{CROSS_PROMO_SNIPPET}")
assert "agent-bench" in CROSS_PROMO_SNIPPET


## Metrics: External Contribution Rate, Plus a Plugin-Specific Number

L60's `Activity` dataclass and `contribution_report()` function are reused unchanged below — the shape of "who filed this, are they a maintainer, how fast did we respond" doesn't change for a plugin project. What's added: a **plugin count** metric that has no equivalent in paper-distiller, because paper-distiller has no plugin mechanism for anyone to contribute to.


In [ ]:
from dataclasses import dataclass
from datetime import date

@dataclass
class Activity:
    date: date
    author: str
    kind: str          # "issue" | "pr" | "plugin_submission"
    is_maintainer: bool
    first_response_hours: float | None = None

# Mock activity log — swap for a real `gh api repos/:owner/:repo/issues` pull once the repo is live.
# Reused shape from L60; "plugin_submission" is a new `kind` value specific to agent-bench.
mock_log = [
    Activity(date(2026, 7, 12), "gouravkhanijoe", "issue", True),
    Activity(date(2026, 7, 12), "gouravkhanijoe", "issue", True),
    Activity(date(2026, 7, 13), "someone_else", "pr", False, first_response_hours=6.0),
    Activity(date(2026, 7, 14), "another_dev", "plugin_submission", False, first_response_hours=18.0),
    Activity(date(2026, 7, 15), "gouravkhanijoe", "issue", True),
]

def contribution_report(log: list[Activity]) -> dict:
    # Reused unchanged from L60 for the core metrics; extended with plugin_count.
    external = [a for a in log if not a.is_maintainer]
    responded = [a for a in external if a.first_response_hours is not None]
    return {
        "total_activity": len(log),
        "external_activity": len(external),
        "external_contribution_rate": round(len(external) / len(log), 2) if log else 0.0,
        "unique_external_contributors": len({a.author for a in external}),
        "avg_first_response_hours": (
            round(sum(a.first_response_hours for a in responded) / len(responded), 1)
            if responded else None
        ),
        # New metric — has no equivalent for paper-distiller (no plugin mechanism there).
        "plugin_count": len([a for a in log if a.kind == "plugin_submission" and not a.is_maintainer]),
    }

report = contribution_report(mock_log)
for k, v in report.items():
    print(f"{k}: {v}")

assert report["plugin_count"] == 1
assert report["external_contribution_rate"] > 0


## 10 Pitfalls in Round-Two OSS Growth

| # | Pitfall | Why it hurts |
|---|---------|---------------|
| 1 | Launching `agent-bench` the same week as a `paper-distiller` push | Splits finite audience attention, weakens both launches |
| 2 | CONTRIBUTING.md documents only "PR to core," omits the plugin path | Steers contributors toward the higher-friction, maintainer-bottlenecked route when a faster one exists |
| 3 | Org-level `.github` repo files drift out of sync with per-repo overrides | Two Codes of Conduct silently disagreeing is worse than one, inconsistently enforced |
| 4 | Trusting a third-party plugin submission without reading its source | Same code-execution risk L66's allowlist and L63's guardrails exist to manage — a plugin request issue is not a security review |
| 5 | README Plugin Gallery lists plugins that were never actually verified to install | A broken "verified" entry destroys trust faster than an empty gallery |
| 6 | One repo's CI badge goes green-but-stale while attention is on the other | Same lying-badge risk as L60 #2, doubled by having two repos to neglect |
| 7 | Cross-promotion link added once, never revisited after either repo's README changes shape | Stale cross-links look like abandonment, not activity |
| 8 | Plugin submission issues filed as generic "feature requests" | No dedicated `plugin_submission.yml` template means gallery entries get lost in the general backlog |
| 9 | Assuming the same `awesome-*` list fits both packages | Wastes the one PR-per-list courtesy; each package needs its own list match |
| 10 | Declaring "production ready" before agent-bench's own CI has run green on a real tag | Same premature-launch risk L64 flagged for paper-distiller, now facing a second package with less track record |


In [ ]:
print("Lesson 69 verification — files generated\n")

expected_repo_files = [
    "README.md",
    "CONTRIBUTING.md",
    "LAUNCH_CHECKLIST.md",
    "labels.yml",
    ".github/PULL_REQUEST_TEMPLATE.md",
    ".github/ISSUE_TEMPLATE/bug_report.yml",
    ".github/ISSUE_TEMPLATE/feature_request.yml",
    ".github/ISSUE_TEMPLATE/plugin_submission.yml",
    ".github/ISSUE_TEMPLATE/config.yml",
]

expected_org_files = [
    "PROFILE_README.md",
    "CODE_OF_CONDUCT.md",
    "SUPPORT.md",
    ".github/FUNDING.yml",
]

all_ok = True
for rel in expected_repo_files:
    p = AGENT_BENCH_ROOT / rel
    ok = p.exists() and p.stat().st_size > 0
    all_ok &= ok
    print(f"  [{'OK' if ok else 'MISSING'}] agent_bench_repo/{rel}  ({p.stat().st_size if p.exists() else 0} bytes)")

for rel in expected_org_files:
    p = ORG_GITHUB_ROOT / rel
    ok = p.exists() and p.stat().st_size > 0
    all_ok &= ok
    print(f"  [{'OK' if ok else 'MISSING'}] dot_github_repo/{rel}  ({p.stat().st_size if p.exists() else 0} bytes)")

assert all_ok, "One or more expected files missing"
assert "Plugin Gallery" in readme_path.read_text()
assert "Path 2: Publish your own plugin" in contributing_path.read_text()
assert report["plugin_count"] == 1
print("\nAll checks passed.")


## Summary

| Concept | What it does |
|---|---|
| Portfolio-level OSS growth | Same funnel as L60, applied across 2+ repos sharing an author identity |
| Org-level `.github` repo | GitHub's fallback for CODE_OF_CONDUCT/SUPPORT/FUNDING when a repo doesn't define its own |
| Organization profile README | `<user>/<user>` repo renders on the GitHub profile — natural cross-promotion surface |
| Plugin Gallery (README section) | Lists built-in + third-party environments, only possible because L66 built real plugin discovery |
| Plugin-shaped contributor funnel | Two paths: PR to core, or publish an independent plugin package (lower friction, no PR needed) |
| `plugin_submission.yml` | Dedicated issue form to catalog third-party plugins, separate from bugs/features |
| Cross-promotion + launch sequencing | Link repos to each other; never hard-launch both in the same week |
| Plugin-count metric | External contribution rate (L60) plus a plugin-specific number with no paper-distiller equivalent |

**Phase 7, Lesson 5 of ~6 (tentative), underway.**

### Homework
1. Actually create the `gouravkhanijoe/.github` repo and the `gouravkhanijoe/gouravkhanijoe` profile repo, push the generated files.
2. Add the cross-promotion snippet to `paper-distiller`'s real README.md.
3. File a real Plugin Submission issue for `agent-bench-shell-plugin` using the new template, to prove the funnel works end to end.
4. Find and PR both packages into their respective `awesome-*` lists (two separate PRs).
5. Draft the one launch post you'll use for `agent-bench` — but don't publish it yet, that's L70.

### Next lesson (L70 preview)

**Launch Day — Phase 7 capstone.** Same shape as L64's `paper-distiller` launch: real GitHub repo, real PyPI release via the OIDC Trusted Publisher runbook from L68, first issues filed, and — new this time — actually executing the cross-promotion and `awesome-*` list submissions drafted today. Closes out Phase 7 at 6 lessons (L65-L70).
